In [0]:
%pip install torch>=2.0.0 torchvision>=0.15.0 xgboost fastf1>=3.6.0

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
from pyspark.sql import SparkSession
from sklearn.metrics import mean_absolute_error
import joblib
import os
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/models

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/dataloader

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/testing

In [0]:
# ================================================================
# FEATURE CONFIGURATION - SINGLE SOURCE OF TRUTH
# ================================================================
# This configuration defines all discrete, continuous, and one-hot encoded features.
# Changes here propagate to ALL models (neural networks AND XGBoost), dataloader, and analysis.
#
# Model-specific usage:
# - Neural Networks (Transformer/LSTM/GRU/CNN-LSTM): Use DISCRETE_FEATURES for embeddings
# - XGBoost: Uses DISCRETE_FEATURES as categorical features (enable_categorical=True)
# - Both: Use CONT_FEATURES + ONEHOT_FEATURES as numerical inputs

# Discrete features: each is a dictionary with:
#   - col: column name in DataFrame
#   - emb_dim: embedding dimension for neural models
#   - zero_based: whether to convert to 0-based indexing (original data 1-based)
DISCRETE_FEATURES = [
    {'col': 'Driver_idx', 'emb_dim': 8, 'zero_based': False},
    #{'col': 'Team_idx', 'emb_dim': 8, 'zero_based': False},
    {'col': 'Compound', 'emb_dim': 4, 'zero_based': True}  # Tyre compound: convert 1..n to 0..n-1
]

# One-hot encoded features: binary indicator columns (0 or 1)
# These are already in binary form and don't need normalization
ONEHOT_FEATURES = [
    # Status indicators (lap validity/conditions)
    'status_1',
    
    # Driver effort flags
    'effort_PUSH', 'effort_CONSERVE', 'Rainfall'
]

# Continuous features (numerical features that get normalized)
CONT_FEATURES = [
    'Circuit_NumberOfTurns', 'Geom_length_official',
    'Geom_altitude', 'Geom_min_corner_radius', 
    'Geom_std_corner_radius', 'Geom_bbox_elongation', 'race_completedness', 'TyreLife', 'AirTemp', 'Humidity', 'WindDirection', 'GapToLeader', 'GapToAhead', 'GapToBehind',
    'TimeSinceLastWeatherMeasurement'
]

# Helper functions
def get_discrete_col_names():
    """Get list of discrete feature column names."""
    return [f['col'] for f in DISCRETE_FEATURES]

def get_discrete_config(col_name):
    """Get configuration for a specific discrete feature."""
    for f in DISCRETE_FEATURES:
        if f['col'] == col_name:
            return f
    return None

def get_onehot_col_names():
    """Get list of one-hot encoded feature column names."""
    return ONEHOT_FEATURES

def get_all_feature_names():
    """Get all feature column names (discrete + continuous + onehot)."""
    return get_discrete_col_names() + CONT_FEATURES + ONEHOT_FEATURES

def compute_embedding_dims(df):
    """Compute embedding dimensions from DataFrame based on DISCRETE_FEATURES config."""
    dims = {}
    for feat in DISCRETE_FEATURES:
        col = feat['col']
        if col in df.columns:
            n_unique = df[col].nunique()
            # For zero-based features, adjust the count
            if feat['zero_based']:
                # Data uses 1..n, we'll convert to 0..n-1
                n_unique = int((df[col] - 1).max() + 1)
            dims[col] = {
                'n_classes': n_unique,
                'emb_dim': feat['emb_dim'],
                'zero_based': feat['zero_based']
            }
    return dims

# ================================================================
# XGBoost-Specific Helpers
# ================================================================
# XGBoost doesn't use embeddings - it treats features differently:
# - Discrete features → categorical (with enable_categorical=True)
# - One-hot features → numerical (0/1 values)
# - Continuous features → numerical

def get_xgboost_categorical_features():
    """Get list of categorical feature names for XGBoost.
    These will be converted to pandas 'category' dtype."""
    return get_discrete_col_names()

def get_xgboost_numerical_features():
    """Get list of numerical feature names for XGBoost.
    This includes both continuous and one-hot encoded features."""
    return CONT_FEATURES + ONEHOT_FEATURES

def get_xgboost_all_features():
    """Get all features for XGBoost (categorical + numerical)."""
    return get_xgboost_categorical_features() + get_xgboost_numerical_features()

In [0]:



# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# 1. Load and preprocess data from Unity Catalog
df = spark.table('workspace.f1_racing_laptime_pred.silver_training').toPandas()
df = assign_stints(df)
df = df[df['dnf']==0]
df = df.drop(columns=['dnf'])

# 2. Handle NaN values in continuous features
# Since data is already normalized (mean=0, std=1), fill NaN with 0 (the mean)
for col in CONT_FEATURES:
    if col in df.columns:
        nan_count = df[col].isna().sum()
        if nan_count > 0:
            print(f"Filling {nan_count} NaN values in {col} with 0")
            df[col] = df[col].fillna(0)

# 3. Use feature configuration from above
cont_features = CONT_FEATURES

# 4. Create dataset and dataloader
dataset = StintDataset(df, cont_features)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

# 5. Compute model parameters from data and configuration
emb_dims = compute_embedding_dims(df)

# Handle optional discrete features (may be commented out in config)
if 'Driver_idx' in emb_dims:
    n_drivers = emb_dims['Driver_idx']['n_classes']
    driver_emb_dim = emb_dims['Driver_idx']['emb_dim']
else:
    n_drivers = 0
    driver_emb_dim = 0

if 'Team_idx' in emb_dims:
    n_teams = emb_dims['Team_idx']['n_classes']
    team_emb_dim = emb_dims['Team_idx']['emb_dim']
else:
    n_teams = 0
    team_emb_dim = 0

if 'Compound' in emb_dims:
    n_tyres = emb_dims['Compound']['n_classes']
    tyre_emb_dim = emb_dims['Compound']['emb_dim']
else:
    n_tyres = 0
    tyre_emb_dim = 0

if 'Circuit_Name' in emb_dims:
    n_circuits = emb_dims['Circuit_Name']['n_classes']
    circuit_emb_dim = emb_dims['Circuit_Name']['emb_dim']
else:
    n_circuits = 0
    circuit_emb_dim = 0

n_cont_features = len(cont_features)

print(f"Model Configuration:")
print(f"  Drivers: {n_drivers} (embedding dim: {driver_emb_dim})")
print(f"  Teams: {n_teams} (embedding dim: {team_emb_dim})")
print(f"  Tyres: {n_tyres} (embedding dim: {tyre_emb_dim})")
print(f"  Circuits: {n_circuits} (embedding dim: {circuit_emb_dim})")
print(f"  Continuous features: {n_cont_features}")

## How to Modify Features

### Adding/Removing Continuous Features:
Edit the `CONT_FEATURES` list above. Changes automatically propagate to:
- Dataset creation
- Model input dimension
- All downstream analysis

### Adding/Removing Discrete Features:
Edit the `DISCRETE_FEATURES` list above. Each entry needs:
- `col`: Column name in your DataFrame
- `emb_dim`: Embedding dimension (typically 4-16, higher for features with more classes)
- `zero_based`: Set `True` if data uses 1-based indexing (will convert to 0-based)

Changes automatically propagate to:
- Model architectures (embedding layers)
- Dataset loader (discrete feature extraction)
- All downstream analysis

**Example: Adding Circuit as a discrete feature**
```python
DISCRETE_FEATURES = [
    {'col': 'Driver_idx', 'emb_dim': 8, 'zero_based': False},
    {'col': 'Team_idx', 'emb_dim': 8, 'zero_based': False},
    {'col': 'Compound', 'emb_dim': 4, 'zero_based': True},
    {'col': 'Circuit_idx', 'emb_dim': 12, 'zero_based': False}  # NEW
]
```

Then update:
1. **dataloader notebook**: StintDataset class to extract the new feature
2. **models notebook**: Model architectures to add the embedding layer and include it in input
3. Re-run this notebook from Cell 5 onwards

### XGBoost Integration:
**XGBoost uses the same feature configuration automatically!**
- `DISCRETE_FEATURES` → treated as categorical (with `enable_categorical=True`)
- `ONEHOT_FEATURES` → used directly as numerical features (0/1 values)
- `CONT_FEATURES` → used directly as numerical features

No separate configuration needed. The XGBoost model class automatically reads from:
- `get_xgboost_categorical_features()` for categorical features
- `get_xgboost_numerical_features()` for continuous + one-hot features
- `get_xgboost_all_features()` for all features

When you add/remove features in this cell, XGBoost picks up the changes automatically.

In [0]:
model = StintTransformer(
    n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres, n_circuits=n_circuits,
    driver_emb_dim=driver_emb_dim, team_emb_dim=team_emb_dim, tyre_emb_dim=tyre_emb_dim, circuit_emb_dim=circuit_emb_dim,
    n_cont_features=n_cont_features
)


In [0]:
model = StintLSTM(
    n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres, n_circuits=n_circuits,
    driver_emb_dim=driver_emb_dim, team_emb_dim=team_emb_dim, tyre_emb_dim=tyre_emb_dim, circuit_emb_dim=circuit_emb_dim,
    n_cont_features=n_cont_features
)

In [0]:
model = StintGRU(
    n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres, n_circuits=n_circuits,
    driver_emb_dim=driver_emb_dim, team_emb_dim=team_emb_dim, tyre_emb_dim=tyre_emb_dim, circuit_emb_dim=circuit_emb_dim,
    n_cont_features=n_cont_features
)

In [0]:
model = StintCNNLSTM(
    n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres, n_circuits=n_circuits,
    driver_emb_dim=driver_emb_dim, team_emb_dim=team_emb_dim, tyre_emb_dim=tyre_emb_dim, circuit_emb_dim=circuit_emb_dim,
    n_cont_features=n_cont_features
)

In [0]:
def train_and_save(model, dataloader, optimizer, criterion, device, num_epochs=19, save_path='stint_transformer_model.pth', use_plateau: bool = True, plateau_kwargs: dict = None):
    """Train provided model and save weights to `save_path`.

    Returns the trained model.
    """
    model.to(device)
    print(f"Starting training for {num_epochs} epochs")
    scheduler = None
    if use_plateau:
        plateau_kwargs = plateau_kwargs or {}
        # sensible defaults: halve LR on plateau, wait 8 epochs
        defaults = dict(mode='min', factor=0.5, patience=8)
        merged = {**defaults, **plateau_kwargs}
        try:
            scheduler = ReduceLROnPlateau(optimizer, **merged)
            print(f"ReduceLROnPlateau scheduler enabled with params: {merged}")
        except TypeError as e:
            # Some torch versions may not support newer kwargs; fall back to a minimal scheduler
            print(f"Warning: ReduceLROnPlateau init failed ({e}); scheduler disabled.")
            scheduler = None
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        valid_batches = 0

        for cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx, lap_time, mask in dataloader:
            # Move to device
            cont_feats = cont_feats.to(device)
            driver_idx = driver_idx.to(device)
            team_idx = team_idx.to(device)
            tyre_idx = tyre_idx.to(device)
            circuit_idx = circuit_idx.to(device) if circuit_idx is not None else None
            lap_time = lap_time.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx, mask=mask)

            # Create mask: exclude padding only (data already filtered for dnf==0)
            valid_mask = ~mask
            
            # Only compute loss on valid (non-padded) laps
            if valid_mask.sum() > 0:  # Ensure there are valid laps in this batch
                loss = criterion(outputs[valid_mask], lap_time[valid_mask])
                
                # Backward pass
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                valid_batches += 1
            # else: skip this batch if no valid laps

        # Average loss over batches that had valid laps
        epoch_metric = epoch_loss / max(valid_batches, 1)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_metric:.4f}")
        # Step scheduler on the monitored metric (training loss here)
        if scheduler is not None:
            scheduler.step(epoch_metric)
            # print current LR(s)
            lrs = {i: g['lr'] for i, g in enumerate(optimizer.param_groups)}
            print(f"Learning rates: {lrs}")

    torch.save(model.state_dict(), save_path)
    model.eval()
    print(f"Model saved to {save_path}")
    return model


In [0]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trained = train_and_save(model, dataloader, optimizer, criterion, device, num_epochs=55, save_path='stint_GRU_model.pth')

In [0]:
import numpy as np
import pandas as pd

# Run predictions on silver_validating and write to gold_predicted_validation
mae, out_df = predict_and_evaluate(
    trained, 
    device, 
    'workspace.f1_racing_laptime_pred.silver_validating',
    scaler_path='/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib'
)

display(out_df['Circuit_Name'].unique())

errors, distribution = print_error_report()

# ================================================================
# DECOMPOSE ERROR: Baseline Pace vs. Within-Session Tracking
# ================================================================
print("\n" + "="*70)
print("ERROR DECOMPOSITION: Baseline Pace vs. Dynamics Tracking")
print("="*70)

# Filter to status_1==1 laps only (matching benchmark filtering)
analysis_df = out_df[out_df['status_1'] == 1].copy()

analysis_df['track_mean'] = (
    analysis_df.groupby(['Circuit_Name'])['actual_sec']
    .transform('mean')
)
analysis_df['track_median'] = (
    analysis_df.groupby(['Circuit_Name'])['actual_sec']
    .transform('median')
)

analysis_df['pred_track_mean'] = (
    analysis_df.groupby(['Circuit_Name'])['pred_sec']
    .transform('mean')
)
analysis_df['pred_track_median'] = (
    analysis_df.groupby(['Circuit_Name'])['pred_sec']
    .transform('median')
)

analysis_df['baseline_discrepancy_mean'] = analysis_df['track_mean'] - analysis_df['pred_track_mean']
print("baseline_mean_discrepancy per track:")
print(analysis_df[["Circuit_Name", "baseline_discrepancy_mean"]].drop_duplicates(subset="Circuit_Name", keep="first"))

analysis_df['baseline_discrepancy_median'] = analysis_df['track_median'] - analysis_df['pred_track_median']
print("baseline_median_discrepancy per track:")
print(analysis_df[["Circuit_Name", "baseline_discrepancy_median"]].drop_duplicates(subset="Circuit_Name", keep="first"))

analysis_df['residual_relative_circuit_mean'] = analysis_df['actual_sec'] - analysis_df['track_mean'] - analysis_df['pred_sec'] + analysis_df['pred_track_mean']
analysis_df['residual_relative_circuit_median'] = analysis_df['actual_sec'] - analysis_df['track_median'] - analysis_df['pred_sec'] + analysis_df['pred_track_median']

print(f"residual_relative_circuit_mean: {analysis_df['residual_relative_circuit_mean'].abs().mean()}")
print(f"residual_relative_circuit_median: {analysis_df['residual_relative_circuit_median'].abs().mean()}")

print(analysis_df['pred_sec'].dtype)
print(analysis_df.groupby('Circuit_Name')['pred_sec'].apply(lambda x: x.dtype))
print(analysis_df.groupby('Circuit_Name')['pred_sec'].agg(['count', 'size']))
print(analysis_df.groupby('Circuit_Name')['actual_sec'].agg(['count', 'size']))





In [0]:
# Evaluate on TRAINING data to check for overfitting
print("Evaluating on TRAINING data...")
mae_train, out_df_train = predict_and_evaluate(
    trained, 
    device, 
    'workspace.f1_racing_laptime_pred.silver_training',
    scaler_path='/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib'
)

print(f"\n{'='*70}")
print("OVERFITTING CHECK")
print(f"{'='*70}")
print(f"Training MAE:   {mae_train:.4f}s")
print(f"Validation MAE: {mae:.4f}s")
print(f"Gap: {mae - mae_train:.4f}s")

if mae - mae_train > 5:
    print("\n⚠️ SEVERE OVERFITTING - model doesn't generalize")
elif mae - mae_train > 2:
    print("\n⚠️ Moderate overfitting")
else:
    print("\n✓ No significant overfitting - likely a data distribution issue")

In [0]:
from sklearn.metrics import mean_squared_error, r2_score
import os
import numpy as np
import torch

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load scalers to convert predictions back to actual seconds
scalers = joblib.load('/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib')
target_mean = scalers['lap_mean']
target_std = scalers['lap_std']
print(f"\nLoaded scalers: mean={target_mean:.4f}, std={target_std:.4f}")

# ============================================
# Benchmark Configuration - Pre-trained Models
# ============================================
models_dir = '/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/models/'

models_to_test = [
    ('LSTM', f'{models_dir}stint_LSTM_model.pth', StintLSTM),
    ('GRU', f'{models_dir}stint_GRU_model.pth', StintGRU),
    ('Transformer', f'{models_dir}stint_transformer_model.pth', StintTransformer),
    ('Transformer_Better', f'{models_dir}stint_transformer_model_better.pth', StintTransformer),
    ('CNN+LSTM', f'{models_dir}stint_CNN+LSTM_model.pth', StintCNNLSTM)
]

results = []

# ============================================
# Load and Evaluate Each Model
# ============================================
for model_name, model_path, ModelClass in models_to_test:
    print("\n" + "="*60)
    print(f"TESTING: {model_name}")
    print("="*60)
    
    # Check if model file exists
    if not os.path.exists(model_path):
        print(f"⚠️  Model file not found: {model_path}")
        print(f"   Skipping {model_name}...")
        continue
    
    print(f"Loading model from: {model_path}")
    
    # Evaluate on validation set using existing predict_and_evaluate function
    try:
        # Instantiate model with correct architecture
        model_instance = ModelClass(
            n_drivers=n_drivers,
            n_teams=n_teams,
            n_tyres=n_tyres,
            n_modes=n_modes,
            n_cont_features=n_cont_features
        )
        
        # Load trained weights
        model_instance.load_state_dict(torch.load(model_path, map_location=device))
        model_instance.to(device)
        model_instance.eval()
        
        # Run predictions (pass model object, not path)
        mae, pred_df = predict_and_evaluate(
            model_instance,
            device,
            'workspace.f1_racing_laptime_pred.silver_validating',
            target_mean=target_mean,
            target_std=target_std
        )
        pred_df = pred_df[pred_df['status_1']==1]
        # Calculate additional metrics from predictions
        # Use rescaled values (actual_sec and pred_sec) if available, otherwise use normalized values
        if 'actual_sec' in pred_df.columns and 'pred_sec' in pred_df.columns:
            y_true = pred_df['actual_sec'].dropna().values
            y_pred = pred_df['pred_sec'].dropna().values
        else:
            y_true = pred_df['actual'].dropna().values
            y_pred = pred_df['pred'].dropna().values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        
        # Store results
        results.append({
            'Model': model_name,
            'MAE (seconds)': mae,
            'RMSE (seconds)': rmse,
            'R²': r2,
            'MAPE (%)': mape
        })
        
        print(f"\n✓ {model_name} Results:")
        print(f"  MAE:   {mae:.4f} seconds")
        print(f"  RMSE:  {rmse:.4f} seconds")
        print(f"  R²:    {r2:.4f}")
        print(f"  MAPE:  {mape:.2f}%")
        
    except Exception as e:
        print(f"❌ Error evaluating {model_name}: {str(e)}")
        continue

# ============================================
# Create Comparison DataFrame
# ============================================
if results:
    comparison_df = pd.DataFrame(results)
    comparison_df = comparison_df.set_index('Model').T
    
    print("\n" + "="*60)
    print("MODEL COMPARISON SUMMARY")
    print("="*60)
    display(comparison_df)
    
    # Show which model performed best
    print("\n" + "="*60)
    print("BEST PERFORMERS")
    print("="*60)
    best_mae_idx = comparison_df.loc['MAE (seconds)'].astype(float).idxmin()
    best_rmse_idx = comparison_df.loc['RMSE (seconds)'].astype(float).idxmin()
    best_r2_idx = comparison_df.loc['R²'].astype(float).idxmax()
    best_mape_idx = comparison_df.loc['MAPE (%)'].astype(float).idxmin()
    
    print(f"Best MAE:   {best_mae_idx} ({comparison_df.loc['MAE (seconds)', best_mae_idx]:.4f}s)")
    print(f"Best RMSE:  {best_rmse_idx} ({comparison_df.loc['RMSE (seconds)', best_rmse_idx]:.4f}s)")
    print(f"Best R²:    {best_r2_idx} ({comparison_df.loc['R²', best_r2_idx]:.4f})")
    print(f"Best MAPE:  {best_mape_idx} ({comparison_df.loc['MAPE (%)', best_mape_idx]:.2f}%)")
else:
    print("\n⚠️  No models were successfully evaluated.")

In [0]:
# ============================================
# XGBoost Training (No Sequences Required)
# ============================================

# Load data directly - no StintDataset/DataLoader needed
df_xgb = spark.table('workspace.f1_racing_laptime_pred.silver_training').toPandas()
df_xgb = assign_stints(df_xgb)

print(f"Loaded {len(df_xgb):,} laps for XGBoost training")

# Define features - Use configuration from Cell 6 (Single Source of Truth)
feature_cols = get_xgboost_all_features()

# Prepare X and y
X_train = df_xgb[feature_cols].copy()
y_train = df_xgb['LapTime_sec'].copy()

# Load validation data
df_val = spark.table('workspace.f1_racing_laptime_pred.silver_validating').toPandas()
df_val = assign_stints(df_val)
X_val = df_val[feature_cols].copy()
y_val = df_val['LapTime_sec'].copy()

print(f"\nTraining: {len(X_train):,} laps")
print(f"Validation: {len(X_val):,} laps")
categorical_features = get_xgboost_categorical_features()
numerical_features = get_xgboost_numerical_features()

print(f"\nFeatures: {len(feature_cols)}")
print(f"  - Categorical: {categorical_features}")
print(f"  - Numerical (Continuous + One-hot): {len(numerical_features)}")

In [0]:
# Create and train XGBoost model
xgb_model = StintXGBoost(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    random_state=42
)

print("Training XGBoost model...\n")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# Save model
xgb_model.save('stint_xgboost.json')
print("\n✓ Model saved to stint_xgboost.json")

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict on validation set
y_pred_train = xgb_model.predict(X_train)
y_pred_val = xgb_model.predict(X_val)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

val_mae = mean_absolute_error(y_val, y_pred_val)
val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
val_r2 = r2_score(y_val, y_pred_val)

print("\n" + "="*50)
print("XGBoost Performance")
print("="*50)
print(f"\nTraining Set:")
print(f"  MAE:  {train_mae:.4f} seconds")
print(f"  RMSE: {train_rmse:.4f} seconds")
print(f"  R²:   {train_r2:.4f}")
print(f"\nValidation Set (Held-out Circuits):")
print(f"  MAE:  {val_mae:.4f} seconds")
print(f"  RMSE: {val_rmse:.4f} seconds")
print(f"  R²:   {val_r2:.4f}")

# Show feature importance
print("\n" + "="*50)
print("Top 15 Most Important Features")
print("="*50)
importance = xgb_model.get_feature_importance(importance_type='gain')
print(importance.head(15))